# EpiCoV 2020

- Number of sequences: `200,524`
- Collection dates: 1 Jan 20 to 6 Jul 20

In [1]:
# General
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Custom helpers
from utils import (standardize_country_name, get_continent, choropleth_continent,
choropleth_world, map_age_to_group, col_fillna, plot_dominant_choropleth, inclusion_exclusion,
standardize_vaccine_status, map_pat_status)

import warnings
warnings.filterwarnings("ignore")

In [2]:
YEAR = "2020"
DATA_PATH = os.path.expanduser(f"~/gcs-data/Full_data/epicov_{YEAR}_200k.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.expanduser(f"~/gcs-data/Full_data/epicov_{YEAR}_200k.parquet")
PLOTS_PATH = f"Nov25_plots/epicov_{YEAR}/"

os.makedirs(PLOTS_PATH, exist_ok=True)

The patient status is varying too much, let's standardize it using the manual annotaion by Dr. Miae Lee.

In [3]:
# For patient status mapping
df_pat = pd.read_excel(os.path.expanduser(f"~/gcs-data/patient_status_mapping_MLedit.xlsx"))
df_pat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pat_stat             117 non-null    object
 1   clinical status      118 non-null    object
 2   hospitalized status  118 non-null    object
 3   severity             118 non-null    object
 4   category             118 non-null    object
 5   remarks              49 non-null     object
 6   further remarks      35 non-null     object
dtypes: object(7)
memory usage: 6.6+ KB


In [4]:
%%time
df = pd.read_csv(DATA_PATH) if DATA_PATH.endswith(".csv") else pd.read_parquet(DATA_PATH)
df.sort_values(by=["Collection date"], inplace=True)
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200524 entries, 8434 to 200523
Data columns (total 18 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   Accession ID                     200524 non-null  object
 1   Submission date                  200524 non-null  object
 2   Virus name                       200524 non-null  object
 3   Collection date                  200524 non-null  object
 4   Location                         200524 non-null  object
 5   Host                             200524 non-null  object
 6   Additional location information  6210 non-null    object
 7   Sampling strategy                11252 non-null   object
 8   Gender                           200524 non-null  object
 9   Patient age                      200519 non-null  object
 10  Patient status                   200509 non-null  object
 11  Last vaccinated                  76 non-null      object
 12  Passage           

((200524, 18), None)

In [5]:
df.nunique()

Accession ID                       200524
Submission date                      1016
Virus name                         200478
Collection date                       188
Location                             4418
Host                                    1
Additional location information      1035
Sampling strategy                     111
Gender                                133
Patient age                           311
Patient status                        133
Last vaccinated                        12
Passage                               103
Specimen                              329
Additional host information           417
Lineage                               946
Clade                                  11
AA Substitutions                    70260
dtype: int64

## Clean the data

Let's fill in missing values, fix spurious values and standardize a few column values

### Gender

In [6]:
df["Gender"].value_counts()

Gender
unknown    126740
Male        38933
Female      34391
49.0           17
unknowm        15
            ...  
64              1
9               1
13              1
62.0            1
23.0            1
Name: count, Length: 133, dtype: int64

In [7]:
a = "100"
a[0].isdigit()

True

In [8]:
[x for x in df["Patient status"].unique() if str(x)[0].isdigit()], [x for x in df["Gender"].unique() if str(x)[0].isdigit()]

([],
 ['3.0',
  '66.0',
  '57.0',
  '69.0',
  '76.0',
  '38.0',
  '63.0',
  '65',
  '58.0',
  '41.0',
  '37.0',
  '34.0',
  '64.0',
  '49.0',
  '31.0',
  '46.0',
  '17.0',
  '39.0',
  '47.0',
  '74.0',
  '42.0',
  '48.0',
  '35.0',
  '79.0',
  '65.0',
  '60.0',
  '33.0',
  '32.0',
  '24.0',
  '68.0',
  '75.0',
  '29.0',
  '52.0',
  '73.0',
  '40.0',
  '26.0',
  '70.0',
  '53.0',
  '22.0',
  '45.0',
  '50.0',
  '92.0',
  '25.0',
  '27.0',
  '21.0',
  '59.0',
  '16.0',
  '41',
  '59',
  '47',
  '46',
  '30',
  '53',
  '25',
  '20',
  '18.0',
  '12',
  '33',
  '55',
  '28.0',
  '36.0',
  '44',
  '20.0',
  '51.0',
  '19.0',
  '51',
  '57',
  '43',
  '52',
  '61',
  '45',
  '27',
  '28',
  '24',
  '35',
  '39',
  '22',
  '32',
  '49',
  '61.0',
  '34',
  '26',
  '75',
  '38',
  '48',
  '58',
  '16',
  '42',
  '37',
  '29',
  '21',
  '72',
  '56',
  '62',
  '40',
  '31',
  '74',
  '14',
  '50',
  '36',
  '63',
  '84.0',
  '54',
  '73',
  '54.0',
  '18',
  '23',
  '43.0',
  '6',
  '17',
  '67

In [9]:
genders = ["male", "unknown", "female"]

indices = df.index[df["Gender"].isin([x for x in df["Gender"].unique() if str(x)[0].isdigit()])].tolist()
print(len(indices))
for i in indices:
    age = df.at[i, "Patient age"]
    if str(age).lower() in genders:
        df.at[i, "Patient age"] = df.at[i, "Gender"]
        df.at[i, "Gender"] = age
df["Gender"].value_counts()

445


Gender
unknown    126872
Male        39090
Female      34546
unknowm        15
0               1
Name: count, dtype: int64

Still one entry is left. Let's see what's the issue.

In [10]:
df[df["Gender"]=="0"]

,Accession ID,Submission date,Virus name,Collection date,Location,Host,Additional location information,Sampling strategy,Gender,Patient age,Patient status,Last vaccinated,Passage,Specimen,Additional host information,Lineage,Clade,AA Substitutions
192363,EPI_ISL_7147154,2021-12-04,hCoV-19/USA/WY-WYPHL-20033054/2020,2020-06-26,North America / USA / Wyoming,Human,NaN,NaN,0,18,unknown,NaN,Original,NaN,NaN,B.1.166,GH,"(Spike_D574Y,NS3_Q57H,NSP12_P323L,Spike_D614G,..."


In [11]:
indices = df.index[df["Gender"].isin(["unknowm", "0"])].tolist()
df.loc[indices, "Gender"] = "unknown"
df["Gender"].value_counts()

Gender
unknown    126888
Male        39090
Female      34546
Name: count, dtype: int64

### Age

In [12]:
import re
from collections import defaultdict

def find_age_formats(df):
    # Identify regex patterns for different Patient age formats and show examples/counts
    ages = pd.Series(df["Patient age"].astype(str).fillna("").values).str.strip()
    unique_ages = pd.Series(ages.unique())

    patterns = {
        "integer_years": r'^\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "decimal_years": r'^\s*\d+\.\d+\s*(?:y|yr|yrs|year|years)?\s*$',
        "months": r'^\s*\d{1,3}\s*(?:m|mo|mos|month|months)\b\.?$',
        "days": r'^\s*\d{1,3}\s*(?:d|day|days)\b\.?$',
        "age_range": r'^\s*(?:<\s*)?\d{1,3}\s*(?:-|–|to)\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "decade_s": r'^\s*\d{2}s\s*$',
        "less_than": r'^\s*[<>]\s*\d{1,3}\s*(?:y|yr|yrs|year|years)?\s*$',
        "birth_year": r'^\s*(?:19|20)\d{2}\s*$',
        "text_labels": r'^\s*(?:newborn|neonate|infant|baby|child|teen(?:ager)?|adolescent|adult|elderly|senior|unknown|unk|n/?a|na|not available)\b',
    }

    compiled = {k: re.compile(v, re.I) for k, v in patterns.items()}

    matches = defaultdict(list)
    for val in unique_ages.dropna().astype(str):
        v = val.strip()
        matched = False
        for name, cre in compiled.items():
            if v and cre.search(v):
                matches[name].append(v)
                matched = True
        if not matched and v:
            matches["unmatched"].append(v)

    # Print summary: count and up to 10 examples for each pattern
    for name in list(compiled.keys()) + ["unmatched"]:
        vals = pd.Series(matches.get(name, [])).drop_duplicates()
        print(f"{name}: {len(vals)} unique matches")
        if len(vals) > 0:
            print(vals.head(10).tolist())
        print("-" * 60)

    # Optional: show top unmatched values to refine regexes
    if matches.get("unmatched"):
        unmatched_counts = ages[ages.isin(matches["unmatched"])].value_counts().head(30)
        print("Top unmatched values (sample counts):")
        print(unmatched_counts)
    
    return matches

In [13]:
matches = find_age_formats(df)

integer_years: 122 unique matches
['73', '56', '72', '32', '35', '27', '50', '45', '36', '30']
------------------------------------------------------------
decimal_years: 72 unique matches
['3.0', '27.0', '19.0', '66.0', '57.0', '69.0', '48.0', '59.0', '0.33', '76.0']
------------------------------------------------------------
months: 19 unique matches
['3 months', '6 months', '4 months', '10 months', '2 months', '7 months', '9 months', '11 months', '5 months', '1 month']
------------------------------------------------------------
days: 4 unique matches
['8 days', '10 days', '21 days', '27 days']
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 6 unique matches
['70s', '80s', '90s', '50s', '20s', '60s']
------------------------------------------------------------
less_than: 12 uniqu

From the above analysis, we see that its either of the following formats:
- `Integer years`: Convert to `float` for consistency
- `Decimal years`: Convert to `float` for consistency
- `Months`: Divide by `12` to get the age in years (1 decimal places)
- `Days`: Since all are less than `30 days` we assign as `0.0` (age in years)
- `Age range`: Assign the mean age (in `age_range` in column, we retain age range and handle standarization)
- `Decade`: Assign the mean age (in `age_range` in column, we retain age range and handle standarization)
- `Greater than`: Assign the lower bound (in `age_range` in column, we retain age range and handle standarization)
- `Text`: They are just different cases of `unknown` so standardize to lower case
- `Unmatched formats`:
    - `nan`, `unkown`, `unknow`: Map to `unknown`
    - Others seem to be like a subtraction operation ==> perform subtraction and have the float value

In [14]:
df[df["Patient age"].isin(["Male", "Female"])]

,Accession ID,Submission date,Virus name,Collection date,Location,Host,Additional location information,Sampling strategy,Gender,Patient age,Patient status,Last vaccinated,Passage,Specimen,Additional host information,Lineage,Clade,AA Substitutions
153446,EPI_ISL_4404950,2021-09-22,hCoV-19/Argentina/PAIS-A0768/2020,2020-06-16,South America / Argentina / Buenos Aires / Ber...,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,NaN,NaN,B.1.499,GH,"(NSP7_S25L,NS3_Q57H,NSP2_T85I,NSP14_A320V,N_S1..."
191444,EPI_ISL_4404965,2021-09-22,hCoV-19/Argentina/PAIS-A0778/2020,2020-06-22,South America / Argentina / Buenos Aires / Ber...,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,NaN,NaN,B.1.499,GH,"(NSP7_S25L,NS3_Q57H,NSP2_T85I,NSP15_S147I,NSP1..."
190334,EPI_ISL_4220000,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-79/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.609,G,"(N_S37P,NSP12_P323L,Spike_D614G,NSP12_Q191R)"
192382,EPI_ISL_4219998,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-77/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.1.222,GR,"(Spike_T732A,N_R203K,N_G204R,NSP15_V172L,NSP13..."
192381,EPI_ISL_4219999,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-78/2020,2020-06-24,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Female,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.369,GH,"(N_S183Y,NS3_Q57H,NSP2_T85I,NSP12_P323L,Spike_..."
143812,EPI_ISL_4220060,2021-09-15,hCoV-19/Mexico/CMX-INMEGEN-04-07-181/2020,2020-07-02,North America / Mexico / Mexico City,Human,NaN,NaN,unknown,Male,unknown,NaN,Original,Oropharyngeal swab,NaN,B.1.189,G,"(NSP15_D128Y,NSP3_C55Y,NSP3_V1229F,NSP12_A449V..."


In [15]:
for i,row in df[df["Patient age"].isin(["Male", "Female"])].iterrows():
    if row["Gender"].lower() in ["nan", "unknown"]:
        df.at[i, "Gender"] = row["Patient age"]
        df.at[i, "Patient age"] = "unknown"

In [16]:
matches1 = find_age_formats(df)

integer_years: 122 unique matches
['73', '56', '72', '32', '35', '27', '50', '45', '36', '30']
------------------------------------------------------------
decimal_years: 72 unique matches
['3.0', '27.0', '19.0', '66.0', '57.0', '69.0', '48.0', '59.0', '0.33', '76.0']
------------------------------------------------------------
months: 19 unique matches
['3 months', '6 months', '4 months', '10 months', '2 months', '7 months', '9 months', '11 months', '5 months', '1 month']
------------------------------------------------------------
days: 4 unique matches
['8 days', '10 days', '21 days', '27 days']
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 6 unique matches
['70s', '80s', '90s', '50s', '20s', '60s']
------------------------------------------------------------
less_than: 12 uniqu

In [17]:
def multi_unit_age(age:str):
    """
    Convert age strings with multiple units (e.g., "1 month 14 days") to age in years (float).
    Assumes 1 year = 12 months, 1 month = 30 days for conversion.
    """
    # Use the combined pattern with OR logic
    pattern = re.compile(
        # 1. Matches 'X years Y months' (looks for 'y' then 'm')
        r'^\s*(?P<years>\d+)\s*y\w*?\s*(?:[,;\-]?\s*)?(?P<val_months>\d+)\s*m\w*?\s*$'
        
        + '|' # OR 
        
        # 2. Matches 'X months Y days' (looks for 'm' then 'd')
        r'^\s*(?P<months>\d+)\s*m\w*?\s*(?:[,;\-]?\s*)?(?P<days>\d+)\s*d\w*?\s*$',
        re.I 
    )

    def calculate_total_years(duration_string):
        match = pattern.search(duration_string)
        
        if not match:
            return -1

        # Use a dictionary to easily check which group was captured (since OR groups return None for the unmatched side)
        data = match.groupdict()
        
        total_years = 0.0

        # --- Case 1: Years/Months Structure Matched ---
        if data['years'] is not None:
            # data['years'] is the first number (X), data['val_months'] is the second (Y)
            Y = int(data['years'])
            M = int(data['val_months'])
            
            # Calculation: Y + M/12
            total_years = Y + (M / 12.0)
            unit = "Years/Months"

        # --- Case 2: Months/Days Structure Matched ---
        elif data['months'] is not None:
            # data['months'] is the first number (X), data['days'] is the second (Y)
            M = int(data['months'])
            D = int(data['days'])
            
            # Calculation: M/12 + D/365.25 (using 365.25 days/year for accuracy)
            total_years = (M / 12.0) + (D / 365.25)
            unit = "Months/Days"
            
        else:
            # Should not happen if one of the OR conditions is met
            return f"Error: Failed to extract values from '{duration_string}'"

        return round(total_years, 1)
    return calculate_total_years(age)

In [18]:
matches = matches1
for i,row in df.iterrows():
    age = str(row["Patient age"]).strip()
    if age in matches["integer_years"] or age in matches["decimal_years"]:
        df.at[i, "Patient age"] = float(re.findall(r'\d+\.?\d*', age)[0])
    elif age in matches["text_labels"]:
        df.at[i, "Patient age"] = "unknown"
    elif age in matches["months"]:
        months = float(re.findall(r'\d+\.?\d*', age)[0])
        df.at[i, "Patient age"] = round(months / 12, 1)
    elif age in matches["days"]:
        days = float(re.findall(r'\d+\.?\d*', age)[0])
        df.at[i, "Patient age"] = 0.0
    elif age in matches["decade_s"]:
        decade = int(re.findall(r'\d{2}', age)[0])
        df.at[i, "Patient age"] = f"{decade}-{decade+9}"
    elif age in matches["less_than"]:
        val = float(age[1:])
        if age.startswith("<"):
            df.at[i, "Patient age"] = val - 5  # Assuming '< X' means X - 5 years
        elif age.startswith(">"):
            df.at[i, "Patient age"] = val + 5  # Assuming '> X' means X + 5 years
    elif age in matches["unmatched"]:
        if age.lower().startswith("u") or age == "nan" or age == 'X':
            df.at[i, "Patient age"] = "unknown"
        elif "-" in age:
            a,b = age.split("-") if "-" in age else age.split("to")
            if a == "2020":
                df.at[i, "Patient age"] = (float(a.strip()) - float(b.strip()))
            else:
                df.at[i, "Patient age"] = (float(a.strip()) + float(b.strip())) / 2
        elif "weeks" in age:
            weeks = float(re.findall(r'\d+\.?\d*', age)[0])
            df.at[i, "Patient age"] = round((weeks * 7) / 365, 1)
        elif "over" in age:
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val + 5  # Assuming 'over X' means X + 5 years
        elif "under" in age:
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val - 5  # Assuming 'under X' means X - 5 years
        elif multi_unit_age(age) != -1:
            val = multi_unit_age(age)
            print(age, val, "Success")
            df.at[i, "Patient age"] = val
        elif multi_unit_age(age) == -1 and re.match(r'^\s*\d{1,3}\s*+', age): # e.g. "90+"
            val = float(re.findall(r'\d+\.?\d*', age)[0])
            print(age, val)
            df.at[i, "Patient age"] = val + 5  # Assuming 'X+' means X + 5 years
    else:
        continue

over 100 100.0
1 month 14 days 0.1 Success
over 100 100.0
17 years 9 months 17.8 Success
90+ 90.0
5 years 4 months 5.3 Success
12 years 6 moths 12.5 Success
90+ 90.0
2 years 11 months 2.9 Success
4 years 6 moths 4.5 Success


In [19]:
find_age_formats(df)

integer_years: 0 unique matches
------------------------------------------------------------
decimal_years: 131 unique matches
['73.0', '56.0', '72.0', '32.0', '35.0', '27.0', '50.0', '45.0', '36.0', '30.0']
------------------------------------------------------------
months: 0 unique matches
------------------------------------------------------------
days: 0 unique matches
------------------------------------------------------------
age_range: 98 unique matches
['55-59', '30-40', '40-50', '60-70', '50-54', '60-64', '70-74', '80-84', '65-69', '31 - 40']
------------------------------------------------------------
decade_s: 0 unique matches
------------------------------------------------------------
less_than: 0 unique matches
------------------------------------------------------------
birth_year: 0 unique matches
------------------------------------------------------------
text_labels: 1 unique matches
['unknown']
------------------------------------------------------------
unmatche

defaultdict(list,
            {'text_labels': ['unknown'],
             'decimal_years': ['73.0',
              '56.0',
              '72.0',
              '32.0',
              '35.0',
              '27.0',
              '50.0',
              '45.0',
              '36.0',
              '30.0',
              '62.0',
              '64.0',
              '39.0',
              '37.0',
              '47.0',
              '44.0',
              '46.0',
              '21.0',
              '61.0',
              '3.0',
              '57.0',
              '68.0',
              '41.0',
              '83.0',
              '29.0',
              '66.0',
              '38.0',
              '15.0',
              '49.0',
              '65.0',
              '25.0',
              '88.0',
              '34.0',
              '52.0',
              '58.0',
              '40.0',
              '24.0',
              '85.0',
              '43.0',
              '67.0',
              '33.0',
              '74.0',
 

In [20]:
df["age_group"] = df["Patient age"].apply(map_age_to_group)
df["age_group"].value_counts()

age_group
unknown    196762
15-49        2103
50-69        1100
>70           548
5-14            7
<5              4
Name: count, dtype: int64

### Additonal host information

Let's see if we can fill in some missing values in other columns using `Additional host information` column

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200524 entries, 8434 to 200523
Data columns (total 19 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   Accession ID                     200524 non-null  object
 1   Submission date                  200524 non-null  object
 2   Virus name                       200524 non-null  object
 3   Collection date                  200524 non-null  object
 4   Location                         200524 non-null  object
 5   Host                             200524 non-null  object
 6   Additional location information  6210 non-null    object
 7   Sampling strategy                11252 non-null   object
 8   Gender                           200524 non-null  object
 9   Patient age                      200524 non-null  object
 10  Patient status                   200509 non-null  object
 11  Last vaccinated                  76 non-null      object
 12  Passage           

In [22]:
sorted([str(x) for x in df["Additional host information"].unique()])

['-',
 '--',
 '01.03.-07.03.2020 skiing in Madona di Campiglio',
 'Active miner',
 'Acute Limphoblastic Leukemia',
 'Acute Lumphoblastic Leukemia',
 'Acute Lymphoblastic Leukemia',
 'Acute Lympoblastic Leukemia',
 'Acute lymphoblastic leukemia',
 'Additional disease: Hypertension',
 'Additional disease: Hypothyroid',
 'Additional diseases: Diabetes and Hypothyroid',
 'Asymptomatic Direct and High risk contact of confirmed case',
 'Austria',
 'Black African American',
 'Black African American, Hispanic',
 'Black African American, Non-Hispanic',
 'Breast cancer',
 'COPD, smoker, dry cough, fever, SPO2 72% on arrival, CXR: B/L infiltrates and pulmonary congestion',
 'COVID-19 positive case who entered from Brazil.',
 'COVID-19 positive case who entered from Canada.',
 'COVID-19 positive case who entered from Egypt.',
 'COVID-19 positive case who entered from France.',
 'COVID-19 positive case who entered from Great Britain (United Kingdom).',
 'COVID-19 positive case who entered from Indi

There are few rows where Gender is missing and this `Additional host information` column has the gender. Let's fix it.

In [23]:
for i,row in df[df["Additional host information"].isin(["Male", "Female"])].iterrows():
    if row["Gender"] == "unknown":
        df.at[i, "Gender"] = row["Additional host information"]

Next, we see a few `'-'`, `'--'`, ``'`'``, `'unknown'`. Let's standardize them to `'unknown'`.

In [24]:
df["Additional host information"].fillna("unknown", inplace=True)

In [25]:
for i,row in df[df["Additional host information"].isin(['-', '--', 'unknown', '`'])].iterrows():
    df.at[i, "Additional host information"] = "unknown"
df[df["Additional host information"].isin(['-', '--', 'unknown', '`'])]["Additional host information"].value_counts()

Additional host information
unknown    199360
Name: count, dtype: int64

In [26]:
sorted([str(x).lower() for x in df["Additional host information"].unique()])

['01.03.-07.03.2020 skiing in madona di campiglio',
 'active miner',
 'acute limphoblastic leukemia',
 'acute lumphoblastic leukemia',
 'acute lymphoblastic leukemia',
 'acute lymphoblastic leukemia',
 'acute lympoblastic leukemia',
 'additional disease: hypertension',
 'additional disease: hypothyroid',
 'additional diseases: diabetes and hypothyroid',
 'asymptomatic direct and high risk contact of confirmed case',
 'atrioventricular connection',
 'austria',
 'black african american',
 'black african american, hispanic',
 'black african american, non-hispanic',
 'breast cancer',
 'cardiopathy',
 'cardiopathy, diabetes',
 'cardiopathy, diabetes, hepatic',
 'cardiopathy, diabetes, obesity',
 'cardiopathy, diabetes, obesity, asthma',
 'cardiopathy, diabetes, renal',
 'case of reinfection, first sample is epi_isl_811148, second sample is epi_isl_811149',
 'case1 day0',
 'case2 day0',
 'case3 day0',
 'cell culture sample',
 'close contact as a care worker',
 'close contact as a home visit 

### Patient status

Many entries seem to have some information about patient status. Let's see if these can be used to fill in missing patient status values.

In [27]:
df["Patient status"].fillna("unknown", inplace=True)
mask = df["Patient status"].astype(str).str.lower().str.startswith("unk")
df.loc[mask, "Patient status"] = "unknown"

First let's see the cases of `"reinfection"`, `"re-infection"`, or `"reinfected"`

In [28]:
# Get unique additional host information values (lowercased) and keep those containing "reinfection"
variations = ["reinfection", "reinfected", "re-infection"]
reinf = [v for v in pd.Series(df["Additional host information"].dropna().astype(str)).str.lower().unique() if any(var in v for var in variations)]
df[df["Additional host information"].str.lower().isin(reinf)]["Patient status"].value_counts()

Patient status
unknown         20
Live             6
Released         4
Reinfection      1
Hospitalized     1
Name: count, dtype: int64

In [29]:
mask = (df["Additional host information"].str.lower().isin(reinf)) & (df["Patient status"].str.lower() == "unknown")
df.loc[mask, "Patient status"] = "Reinfection"
df[df["Additional host information"].str.lower().isin(reinf)]["Patient status"].value_counts()

Patient status
Reinfection     21
Live             6
Released         4
Hospitalized     1
Name: count, dtype: int64

Next, let's match all the variations of patient status in the `Additional host information` column

In [30]:
# Status: 'Outpatient'
op_terms = ['outpatient']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in op_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Outpatient"

# Status: 'asymptomatic direct and high risk contact of confirmed case'
for i,row in df[df["Additional host information"].str.lower().str.contains("asymptomatic direct and high risk contact of confirmed case", na=False)].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Asymptomatic, Suscpected (High Risk Contact)"

# Status: Confirmed/live/Case1
confirmed = ['case1 day0', 'case2 day0', 'case3 day0', 'confirmed', 'patient acquired infection from community', 'patient acquired infection in congregation',
             'first infection', 'family infection', 'family-cluster transmission', 'e.g. patient infected while traveling in ….', 'works at airport, acquired from community',
             'unknown source of infection']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in confirmed])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Infected"
part_match = ['infection', 'infected', 'patient infected', 'infected in', 'imported case from', 'covid-19 positive', 'community case', 'community transmission',]
for i,row in df[df["Additional host information"].str.lower().str.contains("|".join(part_match), na=False)].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Infected"

# Status: EHPAD
ehpad_terms = ['ehpad 5 sens', 'ehpad mas rome']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in ehpad_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "EHPAD"

# Status: Hospitalized
hospitalized_terms = ['hospital', 'hospital general sur de quito']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in hospitalized_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Hospitalized"

# Status: Suspected
suspected_terms = ['close contact as a care worker', 'close contact as a home visit care worker', 'close contact of uc34', 'close contact person', 'come back from wuhan', 'come back from wuhan, china', 'contact', 'contact with imported cases', 'contactant of a traveler returning from dubai',
 'copd, smoker, dry cough, fever, spo2 72% on arrival, cxr: b/l infiltrates and pulmonary congestion', 'exposed donor', 'fever, cough, pneumonia, cxr: bilateral infiltration', 'fever, pneumonia, cxr: bilateral infiltration / retroperotonial hematoma', 'fever, sob, noisy chest, chest pain, cough, spo2 >95% ra, chest: b/l wheeze. cxr: b/l haziness', 'first case of ”community transmission” in the us; reported feb 26, 2020', 'grandson returns from tokyo', 'he traveled to italy (16/02 a 23/02/2020)', 'high risk contact of confirmed case and first pregnant death', 'history of travelling abroad 4 days prior to the start of symptoms', 'history of travelling abroad two weeks prior to the start of symptoms and contact with a suspect covid-19 case at the work environment', 'household contact of uc13 (epi_isl_417317)', 'household contact of workplace case', 'housekeeper', 'husband and wife, no travel history; contact with a confirmed case from ningbo, zhejiang on 1/27', 'hypertensive symptoms of cough and sore throat', 'import italy, sondrio, madesimo 29.2.-.6.3.', 'indian contact of italian tourist', 'italian tourist in india', 'italy skiing with family in trentino vigo di fassa 22-29. february, in contact with previously positive family member', 'large oputbreak in a small village after dancing fest kynice outbreak', 'other: contact of another covid-positive case covid767', 'other: contact of patient sample covid537', 'other: contact of patient sample covid806', 'other: two contacts of this case (covid 1158 and covid1576) tested positive', 'outgoing traveler', 'patient came from italy', 'patient came from osaka', 'patient came from philippine', 'patient came from tokyo', 'patient co-infected with influenza b', 'patient from china (visiting sri lanka)', 'patient from guadalajara', 'patient from wuhan', 'patient from wuhan, china', 'patient had contact with a positive case. symptoms: cough.', 'patient had contact with a positive case. symptoms: fever, general disconfort, headache.', 'patient had contact with a positive case. symptoms: fever, general disconfort, muscle or joint pain, headache, cough.', 'patient has chronic renal problems. symptoms: fever, general disconfort, muscle or joint pain, headache, cough, anosmia, rhinorrhea, dyspnoea.', 'patient has diabetes mellitus. no symptoms.', 'patient return form wuhan, china', 'patient visited the homes of infected people', 'patient was using the same care home as the infected person', 'patient with chronic respiratory diseases', 'patient with severe combined immunodeficiency (scid)', 'probably infected at a concert in the o2 arena - prague', 'probably infected during a holiday in italy; mild respiratory signs', 'severe productive cough, sob, wheeze, spo2 90% in ra, generalized fatigue and tiredness, sore throat, cxr: b/l infiltrates', 'suspect', 'suspected case', 'suspects to be infected in recent trip to guayaquil', 'symptoms begun on 09-03-2020.', 'symptoms: fever, general disconfort, headache, cough, odynophagia, rhinorrhea.', 'symptoms: fever, general disconfort, headache, cough.', 'symptoms: fever, general disconfort, muscle or joint pain, headache, cough, anosmia, rhinorrhea.', 'symptoms: fever, headache, rhinorrhea.', 'symptoms: fever, vomiting or diarrhea, dyspnoea.', 'symptoms: general disconfort, headache, cough.', 'symptoms: general disconfort, muscle or joint pain, headache, cough, anosmia.', 'syntomps begun on 05-03-2020.', 'syntomps begun on 09-03-2020. arrival to mexico on 06-03-20', 'traveled to hubei (1/11 – 1/24); mother and grandmother were confirmed cases', 'traveled to hubei (1/16 – 1/23)']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in suspected_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Suspected"

# Status: Healthcare worker
hcw_terms = ['ehpad mas rome caregiver', 'health care worker', 'health care worker', 'health care worker taking care of uc4 (epi_isl_413561)',
 'health worker', 'health worker', 'healthcare worker', 'healthcare worker first wave', 'healthcare worker infected in hospital', 'healthcare worker: hospital employee', 'healthcare worker: hospital worker', 'hospital staff without symptom']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in hcw_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Healthcare worker"

# Status: Comorbidities
comorb = ['acute limphoblastic leukemia', 'acute lumphoblastic leukemia', 'acute lymphoblastic leukemia', 'acute lymphoblastic leukemia', 'acute lympoblastic leukemia', 'additional disease: hypertension', 'additional disease: hypothyroid', 'additional diseases: diabetes and hypothyroid', 'atrioventricular connection', 'breast cancer', 'cardiopathy', 'cardiopathy, diabetes', 'cardiopathy, diabetes, hepatic', 'cardiopathy, diabetes, obesity', 'cardiopathy, diabetes, obesity, asthma', 'cardiopathy, diabetes, renal', 'diabetes mellitus type 1', 'dm type 2, progressive sob, spo2 88% in ra, high bp, cxr: b/l lung infiltrates and haziness >50% more on the rt side', 'dysmorphological syndrome', 'gastroschisis post surgery', 'glioblastoma', 'lymphoblastic leukemia, diabetes, trisomia 21', 'neogenetic single kindey', 'neprotic syndrome', 'neurological', 'obesity', 'obesity, hypertension, neurological disorder', 'para-umbilical hernia. chest infection, fever 39.5, cough, sob, spo2 85% in ra, blood acidosis, cxr: diffuse b/l infiltrates more in the left side,on mechanical ventilation', 'patient has diabetes mellitus. no symptoms.', 'renal tubular acidosis', 'sensory motor mononeuropathy', 'syphilis', 'thinness', 'xla-immunocompromised patient with prolonged sars-cov-2 infection']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in comorb])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Comorbidities (more info in Additional host information)"

# Status: Pneumonia
pneu_terms = ['pneumonia (chest x-ray, ct)', 'pneumonia / respiratory distress', 'pneumonia / respiratory failure']
for i,row in df[df["Additional host information"].str.lower().isin([x.lower() for x in pneu_terms])].iterrows():
    if row["Patient status"].lower() == "unknown":
        df.at[i, "Patient status"] = "Pneumonia (more info in Additional host information)"

Now, that we have filled in most entries, let's derive four new features from this: `clinical_status`, `hospitalization_status`, `severity`, `who_category`

In [31]:
df = map_pat_status(df, df_pat, "clinical_status")
df = map_pat_status(df, df_pat, "hospitalization_status")
df = map_pat_status(df, df_pat, "severity")
df = map_pat_status(df, df_pat, "who_category")

In [32]:
df["clinical_status"].value_counts(), df["hospitalization_status"].value_counts(), df["severity"].value_counts(), df["who_category"].value_counts()

(clinical_status
 unknown       183261
 active         12841
 recovered       2263
 dead            1119
 suspected        565
 recovered?         6
 Name: count, dtype: int64,
 hospitalization_status
 unknown                                        184327
 no                                               9485
 yes                                              6208
 usually yes - requires higher level of care        18
 usually yes                                        12
 yes by default                                      4
 ususally no                                         1
 Name: count, dtype: int64,
 severity
 unknown                           183805
 mild                                8671
 moderate                            4051
 uninfected                          2269
 dead                                1119
 severe                               118
 unknown - need +RNA to confirm        21
 likely severe?                         1
 Name: count, dtype: int64,
 who_categor

### Location

Split location into `continent`, `country`, `state` and `city`

In [34]:
# Vectorized split and strip (much faster than iterrows)
parts = df["Location"].astype(str).str.split("/", n=3, expand=True)
parts = parts.rename(columns={0: "continent", 1: "country", 2: "state", 3: "city"})
# strip whitespace and replace empty strings with NaN
for c in parts.columns:
    parts[c] = parts[c].where(parts[c].notna(), np.nan).astype(object)
    parts[c] = parts[c].str.strip().replace({"": np.nan})

# Assign new location columns directly (avoid relying on `cols` variable)
df[["continent", "country", "state", "city"]] = parts[["continent", "country", "state", "city"]]
df.drop(columns=["Location"], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200524 entries, 8434 to 200523
Data columns (total 26 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   Accession ID                     200524 non-null  object
 1   Submission date                  200524 non-null  object
 2   Virus name                       200524 non-null  object
 3   Collection date                  200524 non-null  object
 4   Host                             200524 non-null  object
 5   Additional location information  6210 non-null    object
 6   Sampling strategy                11252 non-null   object
 7   Gender                           200524 non-null  object
 8   Patient age                      200524 non-null  object
 9   Patient status                   200524 non-null  object
 10  Last vaccinated                  76 non-null      object
 11  Passage                          200524 non-null  object
 12  Specimen          

Let's see if we can infer more from `Additional location information` column

In [37]:
sorted([str(x).lower() for x in df["Additional location information"].unique()])

['(mc donalds, cínovecká 796, praha 8)',
 '(mcdonald, praha 8)',
 '(the hospital for chronic disease bohnice)',
 '1160.0',
 '1170.0',
 '1180.0',
 '1200.0',
 '1340.0',
 '1970.0',
 '1st settlement',
 '300 bed hospital',
 '3080.0',
 '3090.0',
 '91331',
 '91942',
 '91942.0',
 '92021',
 '92026',
 '92102.0',
 '92103.0',
 '92105.0',
 '92109.0',
 '92110',
 '92110.0',
 '92114',
 '92117',
 '92117.0',
 '92121.0',
 '92126.0',
 '92231',
 '92243',
 'a passenger in nile river cruise ship',
 'accra',
 'ada county',
 'agualva',
 'agualva cacem',
 'aguas livres',
 'alajuela',
 'alajuela, orotina',
 'alajuela, san rafael',
 'algueirão-mem martins',
 'amadora',
 'amajuba',
 'army base camp',
 'aroub',
 'asia / china / wuhan',
 'asia / south korea / gwangju',
 'ausilio mutuo',
 'awa indigenous community',
 'balata',
 'betacoronavirus',
 'bethlehem',
 'bezmialem vakif university hospital',
 'bolzano, italy',
 'bordeaux',
 'buraca',
 'burgenland, austria',
 'cairo',
 'cairo university (abol-reesh) pediatric 

In [41]:
df[df["Additional location information"].str.lower() == 'venice, italy']

,Accession ID,Submission date,Virus name,Collection date,Host,Additional location information,Sampling strategy,Gender,Patient age,Patient status,...,AA Substitutions,age_group,clinical_status,hospitalization_status,severity,who_category,continent,country,state,city
61618,EPI_ISL_419667,2020-04-03,hCoV-19/Austria/CeMM0014/2020,2020-03-13,Human,"Venice, Italy",NaN,Female,50.0,unknown,...,"(NSP3_K19R,NSP12_P323L,Spike_D614G)",unknown,unknown,unknown,unknown,Unknown,Europe,Austria,NaN,NaN


Let's retain the following columns:
- `Accession ID`
- `Collection date`
- `Submission date`
- `Location`
- `Additional location information`
- `Gender`
- `Patient age`
- `Patient status`
- `Last vaccinated`
- `Additional host information`

In [ ]:
cols = ["Accession ID", "Collection date", "Submission date", "Location", "Gender", "Patient status", "Patient age", "Additional location information", "Last vaccinated", "Additional host information"]
df = df[cols]
df.info()

In [ ]:
df["Last vaccinated"].value_counts()

In [ ]:
df[df["Last vaccinated"] == "Suspeito de reinfecção"]

`Suspeito de reinfecção` is Portugese for Suspected Re-infection. So, it can be considered as `Patient status` rather than vaccination status.

In [ ]:
index = df[df["Last vaccinated"] == "Suspeito de reinfecção"].index
df.loc[index, "Patient status"] = "Suspected Re-infection"

Next, let's standardize vaccination status

In [ ]:
df = standardize_vaccine_status(df)
df.shape, df["Last vaccinated"].value_counts()

According to [Ramarao-Milne et. al, 2022](https://www.csbj.org/article/S2001-0370(22)00219-7/fulltext), there is only `0.3%` of meaningful patient data in current available EpiCoV database. Let's see if this applies to our sample as well.

In [ ]:
inclusion_exclusion(df)

Next, let's standardize the patient status

In [ ]:
df["Patient status"] = df["Patient status"].fillna("Unknown")
df.info()

In [ ]:
df["Clinical status"], df["Hospitalization status"], df["Severity"], df["WHO category"] = np.nan, np.nan, np.nan, np.nan
df = map_pat_status(df, df_pat, "Clinical status")
df = map_pat_status(df, df_pat, "Hospitalization status")
df = map_pat_status(df, df_pat, "Severity")
df = map_pat_status(df, df_pat, "WHO category")

In [ ]:
df.info()

## Collection Submission delay statistics

In [ ]:
df["Submission date"] = pd.to_datetime(df["Submission date"], format="mixed")
df["Collection date"] = pd.to_datetime(df["Collection date"], format="mixed")

df["Collection date"].min(), df["Collection date"].max(), df["Submission date"].min(), df["Submission date"].max()

Let us validate if `Submission_date` is after the `Collection_date`

In [ ]:
df[df["Collection date"] > df["Submission date"]].shape

Find average time between collection of sample and submission of sequence collected on or after Jan 2020

In [ ]:
print(f"Number of sequences before 2020: {len(df[df["Collection date"] < pd.to_datetime("2020-01-01")])}")
df = df[df["Collection date"] >= pd.to_datetime("2020-01-01")]
df["collection_to_submission_days"] = (df["Submission date"] - df["Collection date"]).dt.days
df["collection_to_submission_days"].describe()

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["collection_to_submission_days"], bins=50, kde=True, color='blue', stat='density')
sns.lineplot(x=sorted(df["collection_to_submission_days"]), 
             y=np.full(df.shape[0], 0), color='black', linewidth=2)
mean_days = df["collection_to_submission_days"].mean()
plt.axvline(mean_days, color='red', linestyle='--', label=f"Mean = {mean_days:.2f} days")

# Place annotation above the mean line, at the top of the plot
ymax = plt.gca().get_ylim()[1]
plt.annotate(f"Mean = {mean_days:.2f}", xy=(mean_days, ymax*0.7), xytext=(mean_days+0.5, ymax*0.85),
             arrowprops=dict(facecolor='red', arrowstyle='->'), color='red', fontsize=11)

plt.xlabel("Collection to Submission Days (days)")
plt.title(f"Distribution of time between Collection and Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_to_submission_days_distribution_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for submission dates
sns.histplot(df["Submission date"], bins=10, color="orange", label="Submission date", kde=False, alpha=0.4)

# Calculate means
mean_submission = df["Submission date"].mean()

# Plot mean lines
plt.axvline(mean_submission, color="red", linestyle="--", label=f"Mean Submission: {mean_submission.date()}")

# Annotate the bars
ax = plt.gca()
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"submission_dates_spread_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for collection dates
sns.histplot(df["Collection date"], bins=10, color="orange", label="Collection date", kde=False, alpha=0.4)

# Calculate means
mean_collection = df["Collection date"].mean()

# Plot mean lines
plt.axvline(mean_collection, color="red", linestyle="--", label=f"Mean Collection: {mean_collection.date()}")

# Annotate the bars
ax = plt.gca()
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Collection Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_dates_spread_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot histograms for collection and submission dates
sns.histplot(df["Collection date"], bins=10, color="skyblue", label="Collection date", kde=False)
sns.histplot(df["Submission date"], bins=10, color="orange", label="Submission date", kde=False, alpha=0.2)

# Calculate means
mean_collection = df["Collection date"].mean()
mean_submission = df["Submission date"].mean()

# Plot mean lines
plt.axvline(mean_collection, color="blue", linestyle="--", label=f"Mean Collection: {mean_collection.date()}")
plt.axvline(mean_submission, color="red", linestyle="--", label=f"Mean Submission: {mean_submission.date()}")

# Annotate lead time
lead_time = (mean_submission - mean_collection).days
plt.annotate(
    f"Delay in submission: {lead_time} days",
    xy=(mean_submission, plt.ylim()[1]*0.8),
    xytext=(mean_submission, plt.ylim()[1]*0.95),
    arrowprops=dict(facecolor='green', edgecolor='green', arrowstyle='->', lw=2),
    color="black",
    fontsize=14,
    ha='left'
)

plt.xlabel("Date")
plt.ylabel("Count")
plt.title(f"Distribution of Collection and Submission Dates | {YEAR}")
plt.legend()
plt.tight_layout()
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(PLOTS_PATH, f"collection_submission_dates_distribution_{YEAR}.png"), dpi=300)
plt.show()

In [ ]:
df["Gender"].value_counts()

In [ ]:
df["Patient status"].value_counts()

In [ ]:
df["Patient age"].value_counts()

In [ ]:
df[~df["Gender"].isin(["Male", "Female", "unknown"])]

In [ ]:
# Swap the two values in these rows (Patient age and Patent gender)
genders = ["male", "female", "unknown"]
for i, row in df[~df["Gender"].str.lower().isin(genders)].iterrows():
    age = row["Patient age"]
    if age.lower() in genders:
        df.at[i, "Patient age"] = row["Gender"]
        df.at[i, "Gender"] = age

In [ ]:
df["Gender"].value_counts()

In [ ]:
index = 